# Module 1 — Foundations and visual inspection


This is the complete teaching lab for Module 1. Run setup first, then follow the cells in order. Section headings are learning topics; [the current lesson map](../../docs/LESSON_MAP.md) maps them to the course videos. Collapsing a section does not skip execution.

**Practice versus assessment.** Worked examples teach the method; they are not completed capstone evidence. Use the [RiskLens capstone repository](https://github.com/SarasAI-Institute/Building-Custom-AI-Capstone) for exact evidence thresholds and all 24 Met / Not Yet criteria. Synthetic examples, tiny practice splits and unreviewed judge outputs do not establish mastery.

**Runtime.** Read [Colab setup and artifact handoffs](../../docs/COLAB.md). The CPU exercises run separately in `practice.ipynb` and `solutions.ipynb`. This lab uses model downloads and GPU execution. Modules 3–5 require your saved artifacts and explicit train/load choices. A notebook that stops for missing artifacts is not a completed run. See [what was tested](../../docs/VALIDATION.md).


## Shared setup before running

---
### 0 · Setup

We install the few libraries we need. `transformers` gives us models, `datasets` gives us data,
`accelerate` handles device placement, and `scikit-learn` will score our detector at the end.

*(On Colab this takes ~1 minute. The `-q` flag keeps the output quiet.)*

In [ ]:
"""Run this cell first in a fresh Colab session, before importing model libraries."""
import importlib.metadata as _metadata
import platform as _platform
import subprocess as _subprocess
import sys as _sys
import tempfile as _tempfile
from pathlib import Path as _Path

COURSE_PACKAGES = [
    "transformers==5.22.0", "trl==1.17.0", "peft==0.21.2",
    "accelerate==1.15.0", "datasets==5.1.0", "sentence-transformers==6.2.0",
    "faiss-cpu==1.15.1", "num2words==0.5.14",
    "pillow==13.0.0", "scikit-learn==1.9.1", "matplotlib==3.11.2", "requests==2.34.2",
]
if _sys.version_info < (3, 11):
    raise RuntimeError("Use a current Colab Python 3.11+ runtime.")
try:
    _torch_version = _metadata.version("torch")
except _metadata.PackageNotFoundError as _error:
    raise RuntimeError("Use Colab's preinstalled matched PyTorch/torchvision runtime; see docs/COLAB.md.") from _error
_constraints = ["torch==" + _torch_version]
try:
    _constraints.append("torchvision==" + _metadata.version("torchvision"))
except _metadata.PackageNotFoundError:
    raise RuntimeError("The vision labs need a matching torchvision installation; use a fresh Colab runtime.")
if _platform.system() == "Linux":
    COURSE_PACKAGES.append("bitsandbytes==0.50.2")
_before = {}
for _name in ["transformers", "trl", "peft", "datasets", "sentence_transformers", "numpy"]:
    if _name in _sys.modules:
        _before[_name] = getattr(_sys.modules[_name], "__version__", None)
with _tempfile.TemporaryDirectory() as _temporary:
    _constraint_file = _Path(_temporary) / "preserve-colab-torch.txt"
    _constraint_file.write_text("\n".join(_constraints))
    _subprocess.run([_sys.executable, "-m", "pip", "install", "--quiet", "--upgrade-strategy", "only-if-needed",
                     "-c", str(_constraint_file), *COURSE_PACKAGES], check=True)
if _before:
    raise RuntimeError("Packages installed. Restart the Colab session once, then run from the setup cell again before importing models.")
print("Course stack installed; existing PyTorch/torchvision CUDA wheels preserved.")
print("If installation failed, stop here. Start a fresh GPU runtime and inspect the resolver message.")


Quick sanity check — do we actually have a GPU, and how much memory does it have?
If this prints a T4 with ~15 GB, you are exactly where the course expects you to be.

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# T4s (and other pre-Ampere GPUs) have no native bfloat16 — fall back to float16.
DTYPE = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else (torch.float16 if torch.cuda.is_available() else torch.float32)
print("Using dtype:", DTYPE)

## Topic — Welcome & The RiskLens Course Project

**Theory · 6–8 minutes**

Introduces the self-paced course, Met / Not Yet assessment and the continuing RiskLens capstone. Traces the intended flow from product image to defect note, retrieved evidence and Analyst report, and shows how the five module milestones contribute to the final portfolio.



### The continuing RiskLens project

Product image → visual defect note → retrieved reviews and specifications → Analyst report → measured optimization → integrated product demonstration. Each arrow is a component interface to inspect, not a guarantee of correct output.

## Topic — Tokenizers Explained: From Text to Token IDs

**Theory · 6–8 minutes**

Follows one sentence through subword tokenization, token IDs and decoding. Introduces BPE and SentencePiece at a conceptual level, explains special tokens and context budgets, and shows why words, characters and tokens are different units. Multilingual text illustrates how token counts vary.

Run the shared setup first. Keep this tokenizer object for the corresponding topic.

---
### 1 · Foundations refresher: what a Transformer actually does

Before we touch a VLM, let's rebuild the intuition in three tiny steps, using real tools —
not slides. Every modern LLM/VLM does exactly this:

1. **Tokenize** — split text into sub-word pieces and map each to an integer ID.
2. **Embed** — turn each ID into a vector (a point in a high-dimensional space).
3. **Attend** — let every token *look at* every other token and decide what matters.

Step 1 is fully inspectable. Let's look inside a real tokenizer.

In [ ]:
from transformers import AutoTokenizer
from huggingface_hub import model_info
EMBED_MODEL_ID="HuggingFaceTB/SmolLM2-135M-Instruct"
EMBED_REVISION=model_info(EMBED_MODEL_ID).sha
tokenizer=AutoTokenizer.from_pretrained(EMBED_MODEL_ID,revision=EMBED_REVISION)
text="The bottle cap is cracked near the seal."
print("Tokens:",tokenizer.tokenize(text))
print("IDs:",tokenizer.encode(text))


Notice how words get split into sub-word pieces — the model has never seen "words",
only these pieces. **Everything an LLM knows is expressed as operations on these IDs.**

Now step 2: each ID becomes a learned vector. The *meaning* lives in the geometry —
similar concepts end up close together. Let's verify that claim with a tiny experiment.

## Topic — Vocabularies & Token Embeddings

**Theory · 6–8 minutes**

Explains a tokenizer vocabulary as a mapping between token pieces and IDs, then shows the learned embedding table that converts IDs into vectors. Distinguishes input embeddings from contextual representations inside the model and previews why retrieval uses a separate sentence or document embedding model.



In [ ]:
from transformers import AutoModel
import torch.nn.functional as F

model=AutoModel.from_pretrained(EMBED_MODEL_ID,revision=EMBED_REVISION,dtype=torch.float32)
emb = model.get_input_embeddings()          # the embedding table: one vector per token ID

def vec(word):
    ids = tokenizer.encode(word, add_special_tokens=False)
    return emb(torch.tensor(ids)).mean(dim=0)   # average if the word splits into pieces

pairs = [("crack", "fracture"), ("crack", "banana"), ("defect", "flaw")]
for a, b in pairs:
    sim = F.cosine_similarity(vec(a), vec(b), dim=0).item()
    print(f"cos({a!r:12}, {b!r:12}) = {sim:.3f}")

Input embedding rows are trained as part of a language model. Averaging token rows is a demonstration, not a sentence embedding method with a guaranteed synonym ordering. Inspect the observed values; do not promise that crack/fracture outranks crack/banana. Module 2 uses a separately trained retrieval embedding model.

Free the demonstration model before loading the VLM.

In [ ]:
del model, emb
torch.cuda.empty_cache() if torch.cuda.is_available() else None
print("cleaned up")

## Topic — Attention Explained: Queries, Keys & Values

**Theory · 6–8 minutes**

Uses a small sentence diagram to show how queries and keys produce attention weights that combine value vectors. Introduces multiple heads and causal masking, then explains how a token representation changes with context. Keeps the mathematics to one intuitive weighted-sum example.



### Visual companion

Use the attention animation: query–key scores → masked softmax weights → weighted sum of value vectors. A causal decoder cannot attend to future positions. Attention weights alone are not a complete explanation of a model decision.

## Topic — Inside a Transformer

**Theory · 6–8 minutes**

Builds a Transformer block from attention, feed-forward layers, residual connections and normalization. Explains positional information and distinguishes encoder-only, decoder-only and encoder-decoder models. Traces next-token generation through a decoder model so learners can connect the architecture to the inference loop.



### Visual companion

Trace positional information, attention, feed-forward transformations, residual paths and normalization. Keep the decoder generation loop separate from training. Exact normalization placement depends on the architecture.

## Topic — How Vision-Language Models Read Images

**Theory · 6–8 minutes**

Traces a typical VLM from resized image patches through a vision encoder and a connector into the language model. Shows how visual representations interact with text, introduces projection and cross-attention as alternative designs, and connects image resolution to visual token count and computational cost.



### Visual companion

Image preprocessing → visual representations → connector → language model. Projection and cross-attention are different integration designs. Resolution, tiling and model-specific preprocessing determine visual token count; there is no universal tokens-per-image constant.

## Topic — Working with Hugging Face in Colab

**Tutorial: Colab · 6–8 minutes**

Navigates the Hub, checks a model card and license, and loads a compatible checkpoint and tokenizer or processor. Inspects the processed inputs and explains model IDs, revisions and cache reuse. Demonstrates the workflow directly in Colab, with installation and download waits removed.

Record the model card, license, model ID and revision actually used. Cut installation and download waits.

Before running, run Shared setup and inspect CUDA/device/dtype. After the sentiment pipeline, inspect `classifier.tokenizer(reviews, padding=True, return_tensors="pt")` and show input_ids plus attention_mask shapes. After loading the VLM, record `vlm.config._commit_hash` when available and the package versions. The supplied loader does not pin a revision; for reproducible reruns, use the same explicit revision for processor and model. Cache reuse avoids repeat downloads of unchanged cached artifacts; it does not skip model construction or guarantee zero network calls.

---
### 2 · The Hugging Face ecosystem in 5 minutes

Three things to internalize about the [Hub](https://huggingface.co):

| Concept | What it is | Why you care |
|---|---|---|
| **Model repo** | Git repo with weights + config + tokenizer | Matching model and tokenizer/processor classes load their own artifacts |
| **Model card** | The README of a model | License, intended use, eval numbers — *read it before you ship it* |
| **`pipeline()`** | One-liner inference wrapper | A task-specific inference wrapper |

The `pipeline` API is the front door. One line, batteries included:

In [ ]:
from transformers import pipeline

classifier = pipeline("sentiment-analysis",
                      model="distilbert/distilbert-base-uncased-finetuned-sst-2-english")

reviews = [
    "This kettle broke after two days. The lid hinge snapped clean off.",
    "Absolutely love it — heats fast and feels premium.",
]
for r in classifier(reviews):
    print(r)

That's a full inference stack — tokenizer, model, post-processing — in one call.
When you need control (custom prompts, generation settings, chat templates) you drop one level
down to `AutoModel*` classes. That's what we'll do with the VLM.

---
### 3 · Vision-Language Models: giving the Transformer eyes

A **VLM** bolts a vision encoder onto a language model:

```
image → processor → vision encoder → learned connector ─┐
                                            ├──► language model ──► text
text  ──► tokenizer     ──► text tokens  ───┘
```

The processor and vision encoder produce continuous visual features. A learned connector adapts them to the language model interface. Matching dimensions alone does not establish useful grounding; multimodal training supplies that connection. Generated explanations still require task evaluation.

We'll use **SmolVLM2-2.2B-Instruct** — the course checkpoint. Rehearse its memory use on the actual runtime; suitability depends on image processing and generation settings.

> 💡 *Low VRAM (<6 GB)? Swap the model ID for `HuggingFaceTB/SmolVLM-256M-Instruct` — check processor compatibility and revalidate outputs. A model swap changes the experiment.*

In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText
from huggingface_hub import model_info
MODEL_ID = "HuggingFaceTB/SmolVLM2-2.2B-Instruct" if torch.cuda.is_available() else "HuggingFaceTB/SmolVLM-256M-Instruct"
MODEL_REVISION = model_info(MODEL_ID).sha
processor=AutoProcessor.from_pretrained(MODEL_ID,revision=MODEL_REVISION)
vlm=AutoModelForImageTextToText.from_pretrained(MODEL_ID,revision=MODEL_REVISION,
    dtype=DTYPE,device_map={"":0 if torch.cuda.is_available() else "cpu"})
vlm.eval()
print("Loaded", MODEL_ID, MODEL_REVISION, "dtype",DTYPE)
print("The smaller CPU model is a separate demonstration, not a comparable GPU evaluation.")


The **processor** is the multimodal version of a tokenizer: it resizes/normalizes images
*and* tokenizes text, producing one input dict. Conversations use the same chat-message format
you know from LLM APIs — the only new thing is an `image` content block.

Let's write a small helper we'll reuse all session. Note how short it is — the chat template
does the heavy lifting:

## Topic — Run Your First Vision-Language Model

**Tutorial: Colab · 6–8 minutes**

Uses the Week 1 notebook to prepare a product image and text prompt, apply the expected chat template and run generation. Inspects the structured defect output and links the processor inputs to the earlier VLM diagram. Learners repeat the exercise with the remaining images.

Requires processor and vlm from the corresponding topic. This is an inference smoke test, not defect evaluation.

In [ ]:
def ask_vlm(image, question, max_new_tokens=200):
    """Ask the VLM one question about one PIL image. Returns the text answer."""
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": question},
        ],
    }]
    inputs = processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt",
    ).to(vlm.device, dtype=DTYPE)

    with torch.inference_mode():
        out=vlm.generate(**inputs,max_new_tokens=max_new_tokens,do_sample=False)
    new_ids=out[0, inputs["input_ids"].shape[1]:]
    return processor.decode(new_ids,skip_special_tokens=True).strip()


Smoke test with a random image from the web — can it see at all?

In [ ]:
import requests
from PIL import Image
from io import BytesIO

url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/transformers/tasks/australia.jpg"   # public documentation image
response=requests.get(url,timeout=30); response.raise_for_status()
with Image.open(BytesIO(response.content)) as opened: img=opened.convert("RGB")
display(img.resize((320, 240)))

print(ask_vlm(img, "Describe this image in one sentence."))

### Reserve product images for the first demonstration

The cats photo is an optional interface smoke test. The curriculum demonstration below uses bottle images. Reserve one good and one defective image **outside** the first 30 sorted paths per class used by the inherited evaluation adapter. These demo images must never enter the later dev/test pool. Do not tune the prompt here. Run the fixed prompt once, inspect its raw output, and record failures as failures. Downloads and GPU execution still require a clean Colab rehearsal.

---
### 4 · The real task: visual defect detection

**Scenario (this is Capstone Increment 1):** your company manufactures bottles. Quality control
photographs every unit. You must flag defective ones — *and produce a short structured note
per image* that downstream systems (Week 2's RAG, Week 3's Analyst) can consume.

We use **MVTec AD**, the standard industrial anomaly-detection benchmark: real photos of
products with real defects (cracks, contamination, broken parts). We'll work with the
`bottle` category.

*(Dataset mirror on the Hub: `TheoM55/mvtec_anomaly_detection`. Original: mvtec.com — CC BY-NC-SA 4.0,
research/education use.)*

In [ ]:
from huggingface_hub import snapshot_download

# The full mirror is ~5 GB across 15 product categories — we need ONE category.
# snapshot_download with allow_patterns fetches just the bottle folders (~150 MB, ~2 min):
DATA_ROOT = snapshot_download(
    "TheoM55/mvtec_anomaly_detection", repo_type="dataset",
    allow_patterns=["images/train/bottle/*", "images/test/bottle/*"],
)
print("downloaded to:", DATA_ROOT)

> ⚠️ **Know your dataset's conventions before you touch it.** MVTec AD has two quirks that
> would silently break a naive loader:
> 1. The **train split contains only good units** (it was built for anomaly detection, where
>    you train on normal data only). All defective images live in the **test** split.
> 2. The repo also carries **ground-truth segmentation masks** (under `masks/`) — those aren't
>    product photos; our download patterns simply never fetch them.
>
> Note what we did above: we downloaded **files by path pattern**, not "the dataset" — a 5 GB
> corpus became a 150 MB, 2-minute download because we only asked for what we need. The cell
> below is our *data adapter*: a clean `[(PIL image, "good"|"defective"), ...]` interface over
> those files. If you swap mirrors or data sources, only these two cells change — everything
> after them stays the same. That separation between *adapter* and *logic* is a habit worth copying.

In [ ]:
import glob, json
from pathlib import Path

good_demo_candidates = sorted(glob.glob(f"{DATA_ROOT}/images/train/bottle/good/*.png"))
bad_demo_candidates = sorted(p for p in glob.glob(f"{DATA_ROOT}/images/test/bottle/*/*.png") if "/good/" not in p)
assert min(len(good_demo_candidates), len(bad_demo_candidates)) > 30, "Need reserved images beyond the first 30 per class. Inspect this dataset snapshot."
DEMO_IMAGE_PATHS = [good_demo_candidates[-1], bad_demo_candidates[-1]]
BASELINE_POOL_PATHS = good_demo_candidates[:30] + bad_demo_candidates[:30]
assert not set(DEMO_IMAGE_PATHS) & set(BASELINE_POOL_PATHS)
Path("demo_reserved_paths.json").write_text(json.dumps(DEMO_IMAGE_PATHS, indent=2))
product_image = Image.open(DEMO_IMAGE_PATHS[0]).convert("RGB")
display(product_image)


In [ ]:
DEMO_REPORT_PROMPT = (
    'Inspect this glass bottle. Return only a JSON object with keys verdict, defect_type, and description. '
    'verdict must be "GOOD" or "DEFECTIVE". defect_type must be a short string or null. '
    'description must be one sentence describing visible evidence. Do not invent a defect.'
)

def run_product_demo(image):
    messages = [{"role": "user", "content": [
        {"type": "image", "image": image},
        {"type": "text", "text": DEMO_REPORT_PROMPT},
    ]}]
    prepared = processor.apply_chat_template(messages, add_generation_prompt=True,
        tokenize=True, return_dict=True, return_tensors="pt")
    print("Processor inputs:", {k: tuple(v.shape) for k, v in prepared.items() if hasattr(v, "shape")})
    inputs = prepared.to(vlm.device, dtype=DTYPE)
    with torch.inference_mode():
        output = vlm.generate(**inputs, max_new_tokens=120, do_sample=False)
    # This course model is decoder-only: remove the input prefix before decoding.
    generated_ids = output[0, inputs["input_ids"].shape[1]:]
    raw = processor.decode(generated_ids, skip_special_tokens=True).strip()
    print("Raw generated response:", raw)
    try:
        record = json.loads(raw)
        valid = (isinstance(record, dict)
                 and record.get("verdict") in {"GOOD", "DEFECTIVE"}
                 and isinstance(record.get("description"), str)
                 and "defect_type" in record
                 and (record["defect_type"] is None or isinstance(record["defect_type"], str)))
    except json.JSONDecodeError:
        record, valid = None, False
    print("Required structure valid:", valid)
    return {"raw": raw, "parsed": record, "structure_valid": valid}

first_product_demo = run_product_demo(product_image)
# Exercise: repeat the unchanged prompt on DEMO_IMAGE_PATHS[1], not baseline/test images.


A parseable object is not proof of correct inspection. Compare each statement with the displayed image. This function intentionally keeps the raw answer when parsing fails. The original ask_vlm helper remains available for the inherited workshop; its string-based assistant-prefix extraction should be checked against the chosen chat template. The product demonstration above slices generated token IDs explicitly.

## Topic — Freeze the Split & Measure the Baseline

**Tutorial: Colab · 6–8 minutes**

Creates the development/test split before prompt iteration and preserves its version. Scores the baseline, reads the confusion matrix and explains why defect F1 is the capstone comparison metric.

Requires the corresponding topic product data and reserved demo paths. The split cell stores paths, content hashes and membership; reuse its manifest instead of reshuffling. Malformed class responses remain invalid. Report defect F1 with precision, recall, accuracy and invalid rate. This is a custom teaching split, not the official MVTec benchmark protocol.

In [ ]:
import glob
from collections import Counter
from PIL import Image

def extract_items(root, category="bottle", per_class=30):
    """Adapter: [(image, 'good'|'defective'), ...] — good from train, defects from test.
    Plain sorted-glob over the downloaded files: deterministic, no dataset library needed."""
    good_paths = sorted(glob.glob(f"{root}/images/train/{category}/good/*.png"))[:per_class]
    bad_paths = sorted(p for p in glob.glob(f"{root}/images/test/{category}/*/*.png")
                       if "/good/" not in p)[:per_class]
    return ([(Image.open(p).convert("RGB"), "good") for p in good_paths] +
            [(Image.open(p).convert("RGB"), "defective") for p in bad_paths])

items = extract_items(DATA_ROOT)
counts = Counter(lbl for _, lbl in items)
print("class counts:", dict(counts))          # honest reporting — never assume balance
assert counts["good"] >= 10 and counts["defective"] >= 10, 'adapter found too few images — inspect paths'

Always **look at your data** before modeling. Three good units, three defective:

In [ ]:
# Preview only after the train/development/test identities are frozen below.


---
### 5 · Freeze the held-out set — *before* any prompt engineering

This is the moment that separates a demo from an evaluation. We split the data **now**, put the
test set in a drawer, and do all our prompt iteration on the dev set only. If we peek at test
images while tuning prompts, our final number is fiction.

> 🥶 **Frozen means frozen:** fixed random seed, fixed split, and the test loop runs exactly
> once, at the end.

In [ ]:
import random, hashlib, json
from pathlib import Path

# Version the actual files, not only the random seed. Re-running loads the same split.
root = Path(DATA_ROOT)
good_paths = sorted(glob.glob(f"{DATA_ROOT}/images/train/bottle/good/*.png"))[:30]
bad_paths = sorted(p for p in glob.glob(f"{DATA_ROOT}/images/test/bottle/*/*.png") if "/good/" not in p)[:30]
pool = [{"path": str(Path(p).relative_to(root)), "label": label,
         "sha256": hashlib.sha256(Path(p).read_bytes()).hexdigest()}
        for paths, label in [(good_paths, "good"), (bad_paths, "defective")] for p in paths]
assert not set(DEMO_IMAGE_PATHS) & set(good_paths + bad_paths)
manifest_path = Path("inspector_split_v2.json")
pool_digest = hashlib.sha256(json.dumps(pool, sort_keys=True).encode()).hexdigest()
if manifest_path.exists():
    split_manifest = json.loads(manifest_path.read_text())
    assert split_manifest["pool_sha256"] == pool_digest, "Dataset changed; do not silently reuse the old evaluation identity."
else:
    good_order=list(range(30)); bad_order=list(range(30,60))
    random.Random(42).shuffle(good_order); random.Random(43).shuffle(bad_order)
    dev_indices=good_order[:10]+bad_order[:10]
    test_indices=good_order[10:]+bad_order[10:]
    split_manifest = {"version": 2, "seed": 42, "pool_sha256": pool_digest,
        "dataset_snapshot": str(root), "records": pool,
        "dev_indices": dev_indices, "test_indices": test_indices}
    manifest_path.write_text(json.dumps(split_manifest, indent=2))
assert set(split_manifest["dev_indices"]).isdisjoint(split_manifest["test_indices"])
assert sorted(split_manifest["dev_indices"] + split_manifest["test_indices"]) == list(range(len(pool)))
def load_split(indices):
    return [(Image.open(root / pool[i]["path"]).convert("RGB"), pool[i]["label"]) for i in indices]
dev_set = load_split(split_manifest["dev_indices"])
test_set = load_split(split_manifest["test_indices"])
print({"split_version": 2, "pool_sha256": pool_digest,
       "dev_counts": dict(Counter(y for _, y in dev_set)),
       "test_counts": dict(Counter(y for _, y in test_set))})

import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(9, 6))
shown = {"good": 0, "defective": 0}
for img_, label in dev_set:
    row = 0 if label == "good" else 1
    if shown[label] < 3:
        ax = axes[row][shown[label]]
        ax.imshow(img_); ax.set_title(label); ax.axis("off")
        shown[label] += 1
plt.tight_layout(); plt.show()

### Baseline prompt before iteration

Run the fixed GOOD/DEFECTIVE prompt on development images. Keep raw text and invalid responses in the evaluation record. Zero-shot inference still has compute cost; a role instruction is a hypothesis to evaluate, not a guarantee of better behavior.


In [ ]:
PROMPT_V1 = (
    "You are an industrial quality inspector. Look at this product photo. "
    "Answer with exactly one word: GOOD if the product has no visible defects, "
    "or DEFECTIVE if you can see any damage, contamination, or abnormality."
)

def predict(image, prompt=PROMPT_V1):
    answer = ask_vlm(image, prompt, max_new_tokens=10)
    return {"GOOD": "good", "DEFECTIVE": "defective"}.get(answer.strip().upper(), "invalid")

# try it on two dev images
for img_, truth in dev_set[:2]:
    print(f"truth={truth:10}  prediction={predict(img_)}")

Now score it on the **dev set** (never the test set — we're still iterating):

In [ ]:
def summarize_inspector(records):
    n = len(records)
    if not n:
        raise ValueError("Cannot score an empty dataset.")
    tp = sum(r["truth"] == "defective" and r["prediction"] == "defective" for r in records)
    fp = sum(r["truth"] == "good" and r["prediction"] == "defective" for r in records)
    fn = sum(r["truth"] == "defective" and r["prediction"] != "defective" for r in records)
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    matrix = [[sum(r["truth"] == truth and r["prediction"] == pred for r in records)
               for pred in ["good", "defective", "invalid"]] for truth in ["good", "defective"]]
    return {"n": n, "accuracy": sum(r["truth"] == r["prediction"] for r in records) / n,
            "defect_precision": precision, "defect_recall": recall, "defect_f1": f1,
            "invalid_rate": sum(r["prediction"] == "invalid" for r in records) / n,
            "confusion_rows_good_defective_columns_good_defective_invalid": matrix}

def score_inspector(dataset, prompt):
    records = []
    for image, truth in dataset:
        raw = ask_vlm(image, prompt, max_new_tokens=10)
        answer = raw.strip().upper()
        prediction = {"GOOD": "good", "DEFECTIVE": "defective"}.get(answer, "invalid")
        records.append({"truth": truth, "prediction": prediction, "raw": raw})
    return records, summarize_inspector(records)

def show_inspector_confusion(report, title):
    matrix = report["confusion_rows_good_defective_columns_good_defective_invalid"]
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.imshow(matrix, cmap="Blues")
    ax.set_xticks(range(3), ["good", "defective", "invalid"])
    ax.set_yticks(range(2), ["good", "defective"])
    ax.set_xlabel("Prediction"); ax.set_ylabel("Truth"); ax.set_title(title)
    for row in range(2):
        for col in range(3):ax.text(col, row, str(matrix[row][col]), ha="center", va="center", color="black", bbox={"facecolor":"white","alpha":0.8,"edgecolor":"none"})
    plt.tight_layout(); plt.show()

def accuracy(dataset, prompt):
    return sum(predict(image, prompt) == truth for image, truth in dataset) / len(dataset)

baseline_records, baseline_report = score_inspector(dev_set, PROMPT_V1)
acc_v1 = baseline_report["accuracy"]
print(json.dumps(baseline_report, indent=2))
show_inspector_confusion(baseline_report, "Baseline on development images")
Path("inspector_baseline_dev.json").write_text(json.dumps({"split_sha256": pool_digest,
    "model_id": MODEL_ID, "model_revision": MODEL_REVISION,
    "prompt": PROMPT_V1, "records": baseline_records, "metrics": baseline_report}, indent=2))


## Topic — Improve the Inspector & Analyze Its Mistakes

**Tutorial: Colab · 6–8 minutes**

Works through one prompt improvement and a few-shot example using development data. Interprets a test error and one resolution-versus-latency comparison, then points learners to the remaining notebook experiments.



---
### 7 · Iterate on the prompt (dev set only)

Typical failure mode: the model calls tiny reflections "defects" (over-sensitive) or misses
subtle contamination (under-sensitive). We iterate by *telling it what defects look like in
this domain* — cheap domain adaptation, no training:

In [ ]:
PROMPT_V2 = (
    "You are an industrial quality inspector for glass bottles. "
    "Defects include: cracks or chips in the glass, contamination inside the bottle, "
    "and a broken or deformed opening. "
    "Normal reflections and lighting variations are NOT defects. "
    "Answer with exactly one word: GOOD or DEFECTIVE."
)

v2_records, v2_report = score_inspector(dev_set, PROMPT_V2)
acc_v2 = v2_report["accuracy"]
print({"v1_dev": baseline_report, "v2_dev": v2_report})
Path("inspector_prompt_ladder_dev.json").write_text(json.dumps({"split_sha256": pool_digest,
    "v1": {"prompt": PROMPT_V1, "records": baseline_records, "metrics": baseline_report},
    "v2": {"prompt": PROMPT_V2, "records": v2_records, "metrics": v2_report}}, indent=2))


One more rung on the ladder: **few-shot prompting** — instead of *describing* defects, we
*show* the model labeled examples inside the prompt. With a VLM the examples are images:
the message simply interleaves several image blocks with their labels, then the query image.
No training — the "learning" happens in context, and it costs image-tokens on every call:

In [ ]:
def predict_fewshot(image, examples):
    """examples: [(PIL image, 'good'|'defective'), ...] — shown to the model before the query."""
    content = [{"type": "text", "text":
        "You are an industrial quality inspector for glass bottles. Here are labeled examples:"}]
    for ex_img, ex_label in examples:
        content += [{"type": "image", "image": ex_img},
                    {"type": "text", "text": f"Label: {ex_label.upper()}"}]
    content += [{"type": "text", "text": "Now classify this image. Answer GOOD or DEFECTIVE."},
                {"type": "image", "image": image}]

    inputs = processor.apply_chat_template([{"role": "user", "content": content}],
        add_generation_prompt=True, tokenize=True, return_dict=True,
        return_tensors="pt").to(vlm.device, dtype=DTYPE)
    with torch.inference_mode():
        out=vlm.generate(**inputs,max_new_tokens=10,do_sample=False)
    answer=processor.decode(out[0,inputs["input_ids"].shape[1]:],skip_special_tokens=True)
    return {"GOOD": "good", "DEFECTIVE": "defective"}.get(answer.strip().upper(), "invalid")

# One exemplar per class, selected only from development data.
shot_indices = [next(i for i, (_, y) in enumerate(dev_set) if y == label) for label in ["good", "defective"]]
shots = [dev_set[i] for i in shot_indices]
fs_eval_indices = [i for i in range(len(dev_set)) if i not in shot_indices]
fs_eval = [dev_set[i] for i in fs_eval_indices]
fs_records = [{"truth": truth, "prediction": predict_fewshot(image, shots)} for image, truth in fs_eval]
fs_report = summarize_inspector(fs_records)
# Compare both methods on exactly the same non-exemplar images.
v2_same_subset = summarize_inspector([v2_records[i] for i in fs_eval_indices])
acc_fs = fs_report["accuracy"]
print({"v2_same_subset": v2_same_subset, "fewshot_same_subset": fs_report, "exemplar_dev_indices": shot_indices})
Path("inspector_fewshot_dev.json").write_text(json.dumps({"split_sha256": pool_digest,
    "exemplar_dev_indices": shot_indices, "evaluation_dev_indices": fs_eval_indices,
    "v2_metrics": v2_same_subset, "fewshot_metrics": fs_report, "fewshot_records": fs_records}, indent=2))


Few-shot performance is measured on the same non-exemplar development images as its zero-shot comparator. Do not claim that extra image tokens have a universal cost or that few-shot must win. This warm-up keeps few-shot exploratory and chooses between the two zero-shot prompts by development defect F1. The assessed project must justify its own candidate set and compute budget.

The structured note below is a separate interface demonstration. A valid JSON object does not validate its description or establish the accuracy of its verdict.


In [ ]:
import json, re

REPORT_PROMPT = (
    "You are an industrial quality inspector for glass bottles. Inspect this photo and "
    "respond ONLY with JSON in this exact format: "
    '{"verdict": "GOOD" or "DEFECTIVE", "defect_type": "<short label or null>", '
    '"description": "<one sentence>"}'
)

def parse_inspection(raw):
    try:
        note=json.loads(raw)
        if not isinstance(note,dict) or set(note)!={"verdict","defect_type","description"}: raise ValueError("Unexpected schema")
        if note["verdict"] not in {"GOOD","DEFECTIVE"}: raise ValueError("Invalid verdict")
        if note["defect_type"] is not None and not isinstance(note["defect_type"],str): raise ValueError("Invalid defect type")
        if not isinstance(note["description"],str) or not note["description"].strip(): raise ValueError("Description required")
        if note["verdict"]=="DEFECTIVE" and not (isinstance(note["defect_type"],str) and note["defect_type"].strip()): raise ValueError("Defect label required")
        return {**note,"raw":raw,"parse_status":"valid"}
    except (ValueError,TypeError,KeyError) as exc:
        return {"verdict":"PARSE_ERROR","raw":raw,"parse_status":"invalid","error":str(exc)}

def inspect(image):
    return parse_inspection(ask_vlm(image,REPORT_PROMPT,max_new_tokens=120))
print(json.dumps(inspect(dev_set[0][0]),indent=2))


### Input resolution and measured latency

Compare fixed development images at two input sizes before the final test. Record actual processor shapes and image positions: input resizing is not automatically a change in encoder token count. Three repeats and six images are a small paired demonstration. This warm-up keeps the final test at original input size; project budget selection needs its own justified rule.


In [ ]:
import time, statistics

def resolution_trial(image, prompt, longest_edge):
    resized = image.copy(); resized.thumbnail((longest_edge, longest_edge), Image.Resampling.LANCZOS)
    if torch.cuda.is_available(): torch.cuda.synchronize()
    started = time.perf_counter()
    messages = [{"role": "user", "content": [{"type":"image", "image":resized}, {"type":"text", "text":prompt}]}]
    inputs = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt").to(vlm.device, dtype=DTYPE)
    shapes = {k:list(v.shape) for k,v in inputs.items() if hasattr(v,"shape")}
    image_id = getattr(vlm.config, "image_token_id", None)
    image_positions = int((inputs["input_ids"] == image_id).sum().item()) if image_id is not None else None
    with torch.inference_mode(): out = vlm.generate(**inputs, max_new_tokens=10, do_sample=False)
    raw = processor.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    if torch.cuda.is_available(): torch.cuda.synchronize()
    return {"input_size":list(resized.size), "processed_shapes":shapes, "image_positions":image_positions,
        "seconds":time.perf_counter()-started, "raw":raw,
        "prediction":{"GOOD":"good", "DEFECTIVE":"defective"}.get(raw.upper(),"invalid")}

# Small paired demonstration, not a serving benchmark. All images are development images.
resolution_dev_indices = [i for label in ["good", "defective"] for i in [j for j, (_, y) in enumerate(dev_set) if y == label][:3]]
assert len(resolution_dev_indices) == 6, "Need three development images per class for this comparison."
resolution_dev = [dev_set[i] for i in resolution_dev_indices]
resolution_results = []
for edge in [256, 512, 1024]:
    resolution_trial(resolution_dev[0][0], PROMPT_V2, edge)  # warm up this condition
    measured = []
    for i, (image, truth) in enumerate(resolution_dev):
        trials = [resolution_trial(image, PROMPT_V2, edge) for _ in range(3)]
        measured.append({"dev_index":resolution_dev_indices[i], "truth":truth, "prediction":trials[0]["prediction"], "trials":trials})
    metrics = summarize_inspector(measured)
    resolution_results.append({"input_longest_edge":edge, "metrics":metrics, "records":measured,
        "median_request_s":statistics.median(t["seconds"] for row in measured for t in row["trials"])})
Path("inspector_resolution_dev.json").write_text(json.dumps({"split_sha256":pool_digest,
    "prompt":PROMPT_V2, "conditions":resolution_results}, indent=2))
print([{k:v for k,v in row.items() if k!="records"} for row in resolution_results])
# Inspect processed_shapes and image_positions: the processor may resize both inputs similarly.


### Score the frozen test after selection

Commit to the chosen configuration first. Evaluate the baseline and selected candidate under the same frozen protocol, then save their outputs. Reload that evidence for recording; do not repeatedly choose changes based on test results. The matrix keeps invalid outputs visible. Inspect a real error if one occurs.


In [ ]:
import time

# Predeclare selection before reading any test prediction. Ties retain v1.
FINAL_PROMPT = PROMPT_V2 if v2_report["defect_f1"] > baseline_report["defect_f1"] else PROMPT_V1
selection = {"split_sha256": pool_digest, "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "selection_metric": "development defect F1", "candidate_set": "zero-shot v1 and v2",
    "final_prompt": FINAL_PROMPT, "input_resize": "original image; processor defaults", "do_sample": False, "max_new_tokens": 10}
Path("inspector_selected_config.json").write_text(json.dumps(selection, indent=2))
result_path = Path("inspector_frozen_test.json")
if result_path.exists():
    final_record = json.loads(result_path.read_text())
    assert final_record["selection"] == selection, "Stored test evidence belongs to a different frozen configuration. Do not tune against it."
    print("Reusing saved test evidence; no fresh inference.")
else:
    if torch.cuda.is_available(): torch.cuda.synchronize()
    started = time.perf_counter()
    base_test_records, base_test_report = score_inspector(test_set, PROMPT_V1)
    if torch.cuda.is_available(): torch.cuda.synchronize()
    baseline_elapsed = time.perf_counter() - started
    if FINAL_PROMPT == PROMPT_V1:
        final_test_records, final_test_report, selected_elapsed = base_test_records, base_test_report, baseline_elapsed
    else:
        if torch.cuda.is_available(): torch.cuda.synchronize()
        started = time.perf_counter()
        final_test_records, final_test_report = score_inspector(test_set, FINAL_PROMPT)
        if torch.cuda.is_available(): torch.cuda.synchronize()
        selected_elapsed = time.perf_counter() - started
    final_record = {"selection": selection, "baseline": {"records": base_test_records, "metrics": base_test_report},
        "selected": {"records": final_test_records, "metrics": final_test_report, "elapsed_s": selected_elapsed}}
    result_path.write_text(json.dumps(final_record, indent=2))
print(json.dumps({"baseline": final_record["baseline"]["metrics"], "selected": final_record["selected"]["metrics"]}, indent=2))
y_true = [r["truth"] for r in final_record["selected"]["records"]]
y_pred = [r["prediction"] for r in final_record["selected"]["records"]]
sec_per_image = final_record["selected"]["elapsed_s"] / len(test_set)
print("Observed sequential seconds/image:", sec_per_image)
GPU_PRICE_PER_H = None  # Set to your actual provider rate; no invented default price.
if GPU_PRICE_PER_H is not None:
    print("Compute-only estimate per 1k images:", GPU_PRICE_PER_H / 3600 * sec_per_image * 1000)


The timed loop supports an observed sequential seconds-per-image measurement. Cost requires your actual provider rate and billing assumptions; compute-only estimates exclude other costs. Keep timing conditions with the results.


In [ ]:
show_inspector_confusion(final_record["selected"]["metrics"], "Frozen test: selected configuration")
errors = [(i, row) for i, row in enumerate(final_record["selected"]["records"]) if row["truth"] != row["prediction"]]
if errors:
    error_index, error_record = errors[0]
    display(test_set[error_index][0])
    print(error_record)
else:
    print("No observed error on this small test set. Do not invent one; discuss uncertainty and inspect additional frozen cases in a separate study.")


### Interpreting the held-out result

A dev/test gap can reflect overfitting, sampling variation or distribution shift; the gap alone does not prove its cause. Keep test results separate from development tuning. Inspect actual false positives, false negatives and invalid responses. The added resolution cell is a small paired development experiment; the assessed project still needs a justified resolution-budget choice and adequate sample size.


## Topic — Explaining the Module Project: Visual Defect Inspector

**Theory/Demo · 6–8 minutes**

Walks through the Stage 1 workspace and required evidence, distinguishing the warm-up from the learner’s assessed work. Shows what a complete evaluation record contains and explains how feedback identifies a Not Yet criterion.



### Evidence to prepare

Repair the three planted baseline bugs and build the inspector. Preserve the frozen split, measured prompt ladder, defect F1 comparison, error analysis and resolution-budget experiment. Complete D1.1–D1.7 in the capstone notebook.

Completes C1 when all its criteria are Met.

The original capstone starter is preserved under `_sources` for reference. Its legacy points and weekly labels are superseded by this curriculum. Practice measurements are not assessed project results.

### Published competency evidence map

The supplied v2.0 rubric governs these criteria. The tutorial illustrates methods; the learner must supply their own actual project evidence. Review the must-pass criterion first.

| Criterion | Evidence | Met standard |
|---|---|---|
| C1.1 **Must-pass** — Your evaluation split is frozen | D1.1 SPLIT cell; saved seed/counts; pre-iteration notebook snapshot or version history. | Before prompt iteration, freeze at least 40 test images with at least 20 per class and a separate development set. Print seed, membership identifiers and class counts. Preserve a dated snapshot or version record. Use development data for choices and disclose any later test exposure. |
| C1.2 — Your inspector fixes the inherited defects | D1.2 bug diagnoses; D1.3 inspect(); representative outputs and fallback checks. | Diagnose and fix the negation-unsafe parser, context-free prompt and empty defect description. Each diagnosis links a real symptom to its cause and fix. inspect() returns the required JSON fields and valid verdict, uses a domain prompt and structured output, and handles malformed model replies without an uncaught parsing error. Justify the VLM choice. |
| C1.3 — Your prompt iteration is measured | D1.3-D1.4 prompt ladder; saved development outputs. | Compare at least three prompt iterations on the same development set, including a few-shot attempt. Keep every prompt, development accuracy, reason for the change and final selection visible. Include the few-shot token-cost observation. Select the final prompt before final-test scoring. |
| C1.4 — Your performance and error claims are supported | D1.5 images and analysis; D1.6 saved Stage 1 evaluation and ledger. | On the frozen test set, report baseline and final defect F1 and the confusion matrix; final F1 exceeds inspect_v0. Analyze at least three actual errors with plausible hypotheses. If fewer exist, analyze all errors and enough clearly labeled borderline cases to total three. Save the evaluation outputs and explain the baseline recall inflation. |
| C1.5 — Your resource recommendation follows the evidence | D1.6 timing/cost; D1.7 resolution table and recommendation. | Measure development accuracy and seconds per image at three or more image sizes under stated hardware and settings. Record estimated cost per 1,000 inspections with the assumed hourly price. Recommend an image size based on the observed quality/time trade-off and explain what the processor actually does with the resized inputs. |

All criteria must be Met for this competency to be Mastered. See [practice-to-capstone differences](../../docs/ASSESSMENT.md) and the [exact consolidated rubric](https://github.com/SarasAI-Institute/Building-Custom-AI-Capstone/blob/codex/competency-colab-refresh/RUBRICS.md). Preserve full outputs and disclose limitations; no metric compensates for missing required evidence.

This lab uses 40 frozen test images (20 per class), 20 separate development images and three longest-edge settings. Few-shot exemplars are removed from that comparison subset; compare all variants on matching non-exemplar rows for the capstone. At least three real mistakes or borderline cases are required; if fewer mistakes occur, explain actual difficult cases rather than inventing errors.
